# Demo: Transforming Time-Series Data in pandas

We're starting with clean daily bike-share data. The cleaning was done in a previous lesson — deduplication, fixing a unit error, filling gaps. We won't dwell on that here. What we care about now: is this series ready for modeling? Spoiler — it probably isn't, and we need to figure out why and fix it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [ ]:
# --- Cleaning pipeline (pre-run, don't worry about this) ---
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
df = df.groupby("date", as_index=False)["rides"].mean()
df["date"] = pd.to_datetime(df["date"])
mask = (df["date"].dt.year == 2023) & (df["date"].dt.month == 10) & (df["rides"] < 1000)
df.loc[mask, "rides"] = df.loc[mask, "rides"] * 24
df.loc[df["rides"] < 0, "rides"] = np.nan
df = df.set_index("date").asfreq("D")
df["rides"] = df["rides"].interpolate(method="time")
rides = df["rides"]
print(f"Clean series: {len(rides)} days, {rides.index[0].date()} to {rides.index[-1].date()}")

## Rolling statistics — what do they tell us?

Before we run any formal tests, let's just *look* at the data. A rolling mean and rolling standard deviation are the quickest visual check for whether a series is stationary.

Stationary means the statistical properties don't change over time. If the rolling mean drifts up and down, or the rolling std is growing — that's non-stationary. And most classical models need stationarity.

In [ ]:
window = 30
rolling_mean = rides.rolling(window).mean()
rolling_std = rides.rolling(window).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(rides.index, rides.values, color="lightgray", linewidth=0.5, label="Daily")
axes[0].plot(rolling_mean.index, rolling_mean.values, color="black", label=f"{window}-day mean")
axes[0].set_ylabel("Rides")
axes[0].set_title("Rolling Mean")
axes[0].legend()

axes[1].plot(rolling_std.index, rolling_std.values, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("Rolling Std Dev")

plt.tight_layout()
plt.show()

OK look at that rolling mean — it goes up in summer, down in winter, up again. That's a clear seasonal pattern, which means the mean isn't constant. And the std bounces around too. This series is not stationary. We can see it just from the plot.

But let's confirm it with a formal test.

## The ADF test — a formal stationarity check

The Augmented Dickey-Fuller test checks whether a series has a *unit root*, which is statistics-speak for "it wanders and doesn't come back." The null hypothesis is that the series is non-stationary. So:

- **p-value < 0.05** → reject the null → series is stationary
- **p-value >= 0.05** → can't reject → series is non-stationary (or at least, we can't prove it's stationary)

In [ ]:
result = adfuller(rides.dropna(), autolag="AIC")

print(f"ADF statistic: {result[0]:.4f}")
print(f"p-value:       {result[1]:.6f}")
print(f"Lags used:     {result[2]}")
print()
if result[1] < 0.05:
    print("p < 0.05 — the series IS stationary.")
else:
    print("p >= 0.05 — the series is NOT stationary. We need to fix that.")

## Differencing — making it stationary

The standard fix for non-stationarity is *differencing*. Instead of looking at the ride count each day, we look at the **change** from yesterday. That's it — `today - yesterday` for every row.

In pandas, that's just `.diff()`. Let's do it and see what happens.

In [ ]:
rides_diff = rides.diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(rides.index, rides.values, color="black", linewidth=0.5)
axes[0].set_ylabel("Rides")
axes[0].set_title("Original")

axes[1].plot(rides_diff.index, rides_diff.values, color="tab:blue", linewidth=0.5)
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change")
axes[1].set_title("First-differenced")

plt.tight_layout()
plt.show()

See the difference? The original wanders up and down with the seasons. The differenced version bounces around zero — no trend, no drift. That looks stationary.

Let's confirm with the ADF test again.

In [ ]:
result_diff = adfuller(rides_diff.dropna(), autolag="AIC")

print(f"ADF statistic: {result_diff[0]:.4f}")
print(f"p-value:       {result_diff[1]:.6f}")
print()
if result_diff[1] < 0.05:
    print("p < 0.05 — the differenced series IS stationary. We're good.")
else:
    print("Still not stationary. Might need second differencing.")

One round of differencing did the trick. That's the `d=1` in ARIMA(p,d,q) — now you know where it comes from.

## Train/test split — why order matters

Last thing. When you split time-series data for modeling, you **cannot** shuffle randomly. Think about it — if Tuesday's value ends up in training and Monday's in testing, the model has seen the future. That's data leakage.

The rule is simple: everything before the cutoff is training, everything after is testing. Like drawing a line on the timeline.

In [ ]:
test_days = 90
train = rides.iloc[:-test_days]
test = rides.iloc[-test_days:]

print(f"Train: {len(train)} days  ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days  ({test.index[0].date()} to {test.index[-1].date()})")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train.values, color="black", linewidth=0.5, label="Train")
ax.plot(test.index, test.values, color="tab:red", linewidth=0.5, label="Test")
ax.axvline(x=train.index[-1], color="gray", linestyle="--", alpha=0.7)
ax.legend()
ax.set_ylabel("Rides")
ax.set_title("Chronological Train/Test Split")
plt.tight_layout()
plt.show()

Everything to the left of the dashed line is training data. Everything to the right is what we'll evaluate our forecast against. The model never gets to peek at the red part during training.

OK, that's the toolkit: rolling stats to visualize, ADF to test, differencing to fix, and chronological splitting to evaluate. In the exercise, you'll do all of this yourself on the same dataset.